# Pergunta da Visualização
Em quais horas do dia possui uma maior concentração de $CO$ e $NOx$? em cada estação de monitoramento e cada dia da semana. 
Para conseguirmos responder esta pergunta precisaremos realizar seguir os seguintes passos:
* Préprocessamento: 
    * Remoção de colunas que são irrelavantes, sobrando somente os componentes quimicos que estão sendo avaliados em conjunto com estação e data.
    * Correção de valores: 
        * Dados espurios serão excluidos.
        * Dados faltantes serão tratados ou terão sua instancia excluida, para esta visualização acreditamos que excluir as instancias seriia o ideal, como possuimos uma grande quantidade de instancias e pela média que será realizada.
    * Agrupamento:
        * Separar por estação de monitoramento (5 estações medem estes dois compostos)
        * Separar por dia da semana (7 dias)
* Realizar uma média dos valores $CO$ e $NOx$ de instancias com hora igual, criando assim somente uma instancia por hora(24 instancias por grupo)
## Implementar visualização
Os dados estarão distribuidos da seguinte maneira:
| Dias da semena | Estação CA | Estação SP | Estação IR | Estação BG | Estação CG |
| :--- | :--- | :--- | :--- | :--- | :--- |
| Domingo | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |
| Segunda | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |
| Terça | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |
| Quarta | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |
| Quinta | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |
| Sexto | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |
| Sábado | 24 instancias com $CO$ | 24 instancias com $NOx$ | 24 instancias com $CO$ e $NOx$ | 24 instancias com $NOx$ | 24 instancias com $NOx$ |

* Cada grupo presente em uma celula da tabela acima representa um "dia médio" daquela estação, ou seja, domingo na estação CA é referente a uma média de todos os domingos da estação CA, esta média foi realizada utilizando as horas, ou seja, foi feito uma média entre todas as 1h30m de todos os domingos da estação CA.
* Existem diferentes componentes químicos medidos em cada estação, então, na vizualização terá uma opção para escolher se será utilizado os valores de: $CO$, $NO$, $NO_2$ ou $NOx$.
* A vizualização será como uma matriz, cada coluna é referente a uma hora do dia (12:30:00 AM, 1:30:00 AM, 2:30:00 AM, ...)
* A visualização pode ser vista de duas maneiras diferentes, um mesmo dia da semana em varias estações ou a mesma estação com varios dias da semana.
* Cada linha será referente a uma estação ou dia da semana, dependendo de qual das duas maneiras foi escolhida.
* Como podemos ver na tabela presente no README.md, os valores de $CO$ e $NO_2$ podem ser tranformados em um índice de 0 a 400, no qual este possui uma cor referente para cada uma das cinco bandas de qualidade. Para $NO$ e $NOx$ utilizaremos o minimo e máximo dessas variaveis encontrados na base de dados, com estes valores os utilizaremos para realizar um `map` para o valor do índice, para assim também utilizarmos as cores para apresentar os valores.
* Esta vizualização será feita utilizando heatmap

In [1]:
import pandas
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

In [2]:
csv_file = './data_set/data_set_completo.csv'
df = pandas.read_csv(csv_file)

Agora com os imports e com a base de dados carregada, iremos realizar a filtragem para retirar os atributos que não serão utilizados, iremos manter somente os seguintes dados: `data`, `codnum`, `no`, `co`, `no2` e `nox`.

In [3]:
# Variaveis de controle para a visualização
colunas_mantidas = ['data', 'codnum', 'no', 'co', 'no2', 'nox']
estacao_mantidas = [2, 3, 4, 5, 8]
dias_da_semana = [0, 1, 2, 3, 4, 5, 6]
atributos_por_estacao = [["no", "no2", "nox"], ["co"], ["nox", "no", "no2"], ["co", "no", "nox", "no2"], ["no", "no2", "nox"]]
atributos_de_todas_estacao = ['data', 'codnum']
grupo_nox = [0, 2, 3, 4]
grupo_co = [1, 4]
estacoes_usadas = {
    0: "BG",
    1: "CA",
    2: "CG",
    3: "IR",
    4: "SP"
}
for estacao in estacoes_usadas.values():
    os.makedirs(f'./data_set/{estacao}', exist_ok=True)

In [4]:
# Aqui filtramos somente os atributos solicitados e as estações desejadas.

df_filtrado = df[[coluna for coluna in colunas_mantidas if coluna in df.columns]].copy()
print(df_filtrado.size)
df_filtrado = df_filtrado[df_filtrado['codnum'].isin(estacao_mantidas)].copy()
print(df_filtrado.size)
df_filtrado.head()

5862276
3654012


,data,codnum,no,co,no2,nox
0,1/1/2011 3:30:00 AM,3,NaN,0.17,NaN,NaN
1,1/1/2011 4:30:00 AM,3,NaN,0.21,NaN,NaN
2,1/1/2011 5:30:00 AM,3,NaN,0.17,NaN,NaN
3,1/1/2011 6:30:00 AM,3,NaN,0.22,NaN,NaN
4,1/1/2011 7:30:00 AM,3,NaN,0.22,NaN,NaN


In [5]:
# Salva o df_filtrado como csv para analise visual
df_filtrado.to_csv('./data_set/df_filtrado.csv', index=False)

In [6]:
# Aqui agrupamos o df_filtrado por dia da semana e estação, para facilitar a visualização
# Também realizamos a ultima filtragem tirando instancias com dados faltando, para evitar erros na visualização
df_filtrado_agrupado = {}
for dia in range(len(dias_da_semana)):
    df_filtrado_agrupado[dia] = {}
    for estacao in range(len(estacao_mantidas)):
        df_filtrado_agrupado[dia][estacao] = df_filtrado[(df_filtrado['codnum'] == estacao_mantidas[estacao]) & (pandas.to_datetime(df_filtrado['data']).dt.dayofweek == dia)].copy()
        # Também realizamos a ultima filtragem tirando instancias com dados faltando, para evitar erros na visualização
        df_filtrado_agrupado[dia][estacao] = df_filtrado_agrupado[dia][estacao].dropna(subset=atributos_de_todas_estacao + atributos_por_estacao[estacao]).copy()
        df_filtrado_agrupado[dia][estacao] = df_filtrado_agrupado[dia][estacao][atributos_de_todas_estacao + atributos_por_estacao[estacao]].copy()

        print(f'Estação {estacao_mantidas[estacao]} - 1/1/2011 3:30:00 AM,3,Dia da semana {dia} - Tamanho do DataFrame: {df_filtrado_agrupado[dia][estacao].size}')

Estação 2 - 1/1/2011 3:30:00 AM,3,Dia da semana 0 - Tamanho do DataFrame: 68550
Estação 3 - 1/1/2011 3:30:00 AM,3,Dia da semana 0 - Tamanho do DataFrame: 49443
Estação 4 - 1/1/2011 3:30:00 AM,3,Dia da semana 0 - Tamanho do DataFrame: 72190
Estação 5 - 1/1/2011 3:30:00 AM,3,Dia da semana 0 - Tamanho do DataFrame: 80202
Estação 8 - 1/1/2011 3:30:00 AM,3,Dia da semana 0 - Tamanho do DataFrame: 57475
Estação 2 - 1/1/2011 3:30:00 AM,3,Dia da semana 1 - Tamanho do DataFrame: 68775
Estação 3 - 1/1/2011 3:30:00 AM,3,Dia da semana 1 - Tamanho do DataFrame: 49581
Estação 4 - 1/1/2011 3:30:00 AM,3,Dia da semana 1 - Tamanho do DataFrame: 71235
Estação 5 - 1/1/2011 3:30:00 AM,3,Dia da semana 1 - Tamanho do DataFrame: 80496
Estação 8 - 1/1/2011 3:30:00 AM,3,Dia da semana 1 - Tamanho do DataFrame: 57735
Estação 2 - 1/1/2011 3:30:00 AM,3,Dia da semana 2 - Tamanho do DataFrame: 68855
Estação 3 - 1/1/2011 3:30:00 AM,3,Dia da semana 2 - Tamanho do DataFrame: 49719
Estação 4 - 1/1/2011 3:30:00 AM,3,Dia da

In [7]:
df_filtrado_agrupado[0][4].head(24)

,data,codnum,no,no2,nox
725284,10/1/2012 3:30:00 PM,8,11.24,31.92,43.16
725285,10/1/2012 4:30:00 PM,8,9.99,37.55,47.54
725286,10/1/2012 5:30:00 PM,8,6.25,30.36,36.61
725287,10/1/2012 6:30:00 PM,8,7.01,26.28,33.30
725288,10/1/2012 7:30:00 PM,8,8.29,29.26,37.55
725289,10/1/2012 8:30:00 PM,8,7.88,33.64,41.52
725290,10/1/2012 9:30:00 PM,8,10.07,29.73,39.80
725291,10/1/2012 10:30:00 PM,8,21.27,31.85,53.12
725292,10/1/2012 11:30:00 PM,8,11.48,29.16,40.65
725437,10/8/2012 12:30:00 AM,8,0.22,8.50,8.21


In [8]:
# Realizamos uma tranformação no atributo data para um novo atributo chamado hora, O alterando para o utilizar 
# o padrão de 24 horas e retirando a parte referente ao dia ex: "1/1/2012 12:30:00 AM" -> "00:30:00", 
# "1/3/2011 1:30:00 AM" -> "01:30:00", "1/3/2011 1:30:00 PM" -> "13:30:00"
# A hora será utilizada mais adiante para realizar a média de todas as instâncias com hora igual.
df_tranformado_agrupado = {}
for dia in range(len(dias_da_semana)):
    df_tranformado_agrupado[dia] = {}
    for estacao in range(len(estacao_mantidas)):
        df_tranformado_agrupado[dia][estacao] = df_filtrado_agrupado[dia][estacao].copy()

        df_tranformado_agrupado[dia][estacao]['hora'] = pandas.to_datetime(
            df_tranformado_agrupado[dia][estacao]['data'], 
            format='%m/%d/%Y %I:%M:%S %p'
        ).dt.strftime('%H:%M:%S')
        
        df_tranformado_agrupado[dia][estacao] = df_tranformado_agrupado[dia][estacao].drop(columns=['data'])


In [9]:
df_tranformado_agrupado[0][4].head(24)

,codnum,no,no2,nox,hora
725284,8,11.24,31.92,43.16,15:30:00
725285,8,9.99,37.55,47.54,16:30:00
725286,8,6.25,30.36,36.61,17:30:00
725287,8,7.01,26.28,33.30,18:30:00
725288,8,8.29,29.26,37.55,19:30:00
725289,8,7.88,33.64,41.52,20:30:00
725290,8,10.07,29.73,39.80,21:30:00
725291,8,21.27,31.85,53.12,22:30:00
725292,8,11.48,29.16,40.65,23:30:00
725437,8,0.22,8.50,8.21,00:30:00


In [10]:
# Realizamos o calculo da média e desvio padrão de todas as instâncias com datetime igual, para cada estação e dia da semana.
df_media = {}
df_desvio_padrao = {}
for dia in range(len(dias_da_semana)):
    df_media[dia] = {}
    df_desvio_padrao[dia] = {}
    for estacao in range(len(estacao_mantidas)):
        df_media[dia][estacao] = df_tranformado_agrupado[dia][estacao].groupby('hora').mean().reset_index()
        df_desvio_padrao[dia][estacao] = df_tranformado_agrupado[dia][estacao].groupby('hora').std().reset_index()
        print(f'Estação {estacao_mantidas[estacao]} - Dia da semana {dia} - Tamanho do DataFrame: {df_media[dia][estacao].size}')
#df_media[0][0].head(24)
df_desvio_padrao[0][0].head(24)

Estação 2 - Dia da semana 0 - Tamanho do DataFrame: 120
Estação 3 - Dia da semana 0 - Tamanho do DataFrame: 72
Estação 4 - Dia da semana 0 - Tamanho do DataFrame: 120
Estação 5 - Dia da semana 0 - Tamanho do DataFrame: 144
Estação 8 - Dia da semana 0 - Tamanho do DataFrame: 120
Estação 2 - Dia da semana 1 - Tamanho do DataFrame: 120
Estação 3 - Dia da semana 1 - Tamanho do DataFrame: 72
Estação 4 - Dia da semana 1 - Tamanho do DataFrame: 120
Estação 5 - Dia da semana 1 - Tamanho do DataFrame: 144
Estação 8 - Dia da semana 1 - Tamanho do DataFrame: 120
Estação 2 - Dia da semana 2 - Tamanho do DataFrame: 120
Estação 3 - Dia da semana 2 - Tamanho do DataFrame: 72
Estação 4 - Dia da semana 2 - Tamanho do DataFrame: 120
Estação 5 - Dia da semana 2 - Tamanho do DataFrame: 144
Estação 8 - Dia da semana 2 - Tamanho do DataFrame: 120
Estação 2 - Dia da semana 3 - Tamanho do DataFrame: 120
Estação 3 - Dia da semana 3 - Tamanho do DataFrame: 72
Estação 4 - Dia da semana 3 - Tamanho do DataFrame: 

,hora,codnum,no,no2,nox
0,00:30:00,0.0,4.966916,13.733983,16.698369
1,01:30:00,0.0,4.208124,12.860494,15.226974
2,02:30:00,0.0,3.841690,11.539158,13.637596
3,03:30:00,0.0,3.270227,10.926027,12.614388
4,04:30:00,0.0,3.399875,10.440771,12.317937
5,05:30:00,0.0,5.503769,10.007547,13.293568
6,06:30:00,0.0,9.640984,10.403171,16.920071
7,07:30:00,0.0,15.822076,11.671653,24.161907
8,08:30:00,0.0,15.249635,14.403787,27.178568
9,09:30:00,0.0,10.296759,14.647381,23.210336


In [11]:
for dia in range(len(dias_da_semana)):
    for estacao in range(len(estacao_mantidas)):
        df_media[dia][estacao].to_csv(f'./data_set/{estacoes_usadas[estacao]}/df_media_{dias_da_semana[dia]}.csv', index=False)
        df_desvio_padrao[dia][estacao].to_csv(f'./data_set/{estacoes_usadas[estacao]}/df_desvio_padrao_{dias_da_semana[dia]}.csv', index=False)

In [12]:
valores_maximos = {
    'co': 0.0,
    'no': 0.0,
    'no2': 0.0,
    'nox': 0.0
}
valores_maximos_desvio_padrao = {
    'co': 0.0,
    'no': 0.0,
    'no2': 0.0,
    'nox': 0.0
}
valores_minimos = {
    'co': float('inf'),
    'no': float('inf'),
    'no2': float('inf'),
    'nox': float('inf')
}
valores_minimos_desvio_padrao = {
    'co': float('inf'),
    'no': float('inf'),
    'no2': float('inf'),
    'nox': float('inf')
}
estacoes_usadas = {
    'co': [1, 4],
    'no': [0, 2, 3, 4],
    'no2': [0, 2, 3, 4],
    'nox': [0, 2, 3, 4]
}

for atributo in valores_maximos:
    for dia in range(len(dias_da_semana)):
        for estacao in estacoes_usadas[atributo]:
            df_estacao = df_media[dia][estacao]
            df_estacao_desvio_padrao = df_desvio_padrao[dia][estacao]
            if atributo not in df_estacao.columns:
                continue
            if atributo not in df_estacao_desvio_padrao.columns:
                continue
            valor_maximo = df_estacao[atributo].max()
            valor_minimo = df_estacao[atributo].min()
            valor_maximo_desvio_padrao = df_estacao_desvio_padrao[atributo].max()
            valor_minimo_desvio_padrao = df_estacao_desvio_padrao[atributo].min()

            if pandas.notna(valor_maximo):
                valores_maximos[atributo] = max(valores_maximos[atributo], valor_maximo)

            if pandas.notna(valor_minimo):
                valores_minimos[atributo] = min(valores_minimos[atributo], valor_minimo)

            if pandas.notna(valor_maximo_desvio_padrao):
                valores_maximos_desvio_padrao[atributo] = max(valores_maximos_desvio_padrao[atributo], valor_maximo_desvio_padrao)

            if pandas.notna(valor_minimo_desvio_padrao):
                valores_minimos_desvio_padrao[atributo] = min(valores_minimos_desvio_padrao[atributo], valor_minimo_desvio_padrao)

print("Valores máximos:", valores_maximos)
print("Valores mínimos:", valores_minimos)
print("Valores máximos desvio padrão:", valores_maximos_desvio_padrao)
print("Valores mínimos desvio padrão:", valores_minimos_desvio_padrao)

Valores máximos: {'co': np.float64(0.6102873563218391), 'no': np.float64(56.16928446771379), 'no2': np.float64(53.31723958333333), 'nox': np.float64(100.13852686308492)}
Valores mínimos: {'co': np.float64(0.24517191977077363), 'no': np.float64(2.4862822719449227), 'no2': np.float64(11.235817555938038), 'nox': np.float64(13.700051635111876)}
Valores máximos desvio padrão: {'co': np.float64(0.4129558823639191), 'no': np.float64(68.46512213221507), 'no2': np.float64(29.611740381327564), 'nox': np.float64(78.24163600449413)}
Valores mínimos desvio padrão: {'co': np.float64(0.1463218547545294), 'no': np.float64(1.585858531069313), 'no2': np.float64(6.777459613293814), 'nox': np.float64(7.179511569503847)}


In [ ]:
from IPython.display import HTML
estacao_mantidas = [2, 3, 4, 5, 6]
dias_da_semana = [0, 1, 2, 3, 4, 5, 6]
atributos_por_estacao = [["no", "no2", "nox"], ["co"], ["nox", "no", "no2"], ["co", "no", "nox", "no2"], ["no", "no2", "nox"]]
estacoes = {
    0: "Praça Cardeal Arco Verde / Copacabana (AV)",
    1: "Bangu (BG)",
    2: "Largo da Carioca / Centro (CA)",
    3: "Campo Grande (CG)",
    4: "Irajá (IR)",
    5: "Pedra de Guaratiba (PG)",
    6: "São Cristóvão (SC)",
    7: "Praça Sans Peña / Tijuca (SP)"
}
estacoes_usadas = {
    0: "BG",
    1: "CA",
    2: "CG",
    3: "IR",
    4: "SP"
}

# pandas dayofweek: 0=segunda, 6=domingo
dias_semana = {
    0: "Segunda",
    1: "Terça",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sábado",
    6: "Domingo"
}

horas_ordem = [f"{h:02d}:30:00" for h in range(24)]

COLOR_SCALE_HEATMAP = [
    [0.0, "#00E400"],
    [0.25, "#FFFF00"],
    [0.5, "#FF7E00"],
    [0.75, "#FF0000"],
    [1.0, "#8F3F97"],
]


def extrair_serie(df_tmp, variavel):
    if variavel in df_tmp.columns:
        return df_tmp[variavel]

    return pandas.Series([float("nan")] * len(df_tmp), index=df_tmp.index, name=variavel)


def obter_config_cores(variavel, metrica):
    if metrica == "Desvio padrão":
        return (
            COLOR_SCALE_HEATMAP,
            valores_minimos_desvio_padrao.get(variavel),
            valores_maximos_desvio_padrao.get(variavel),
        )

    return COLOR_SCALE_HEATMAP, valores_minimos.get(variavel), valores_maximos.get(variavel)


def montar_heatmap(modo, variavel, contexto, metrica):
    colorscale, zmin, zmax = obter_config_cores(variavel, metrica)
    base_dados = df_desvio_padrao if metrica == "Desvio padrão" else df_media

    if zmin is None or zmax is None or pandas.isna(zmin) or pandas.isna(zmax):
        zmin, zmax = 0, 1

    if zmin == zmax:
        zmax = zmin + 1e-9

    if modo == "Estações por dia":
        dia = contexto
        dados = []
        labels_y = []

        for estacao_idx, nome_estacao in estacoes_usadas.items():
            df_tmp = base_dados[dia][estacao_idx].copy()
            df_tmp = df_tmp.set_index("hora").reindex(horas_ordem)

            if variavel not in df_tmp.columns:
                continue

            serie = extrair_serie(df_tmp, variavel)
            dados.append(serie.tolist())
            labels_y.append(nome_estacao)

        titulo = f"{metrica} de {variavel.upper()} por hora - {dias_semana[dia]} - comparação entre estações"
        eixo_y = "Estação"

    else:
        estacao = contexto
        dados = []
        labels_y = []

        for dia_idx in range(7):
            df_tmp = base_dados[dia_idx][estacao].copy()
            df_tmp = df_tmp.set_index("hora").reindex(horas_ordem)

            if variavel not in df_tmp.columns:
                continue

            serie = extrair_serie(df_tmp, variavel)
            dados.append(serie.tolist())
            labels_y.append(dias_semana[dia_idx])

        titulo = f"{metrica} de {variavel.upper()} por hora - {estacoes_usadas[estacao]} - comparação entre dias da semana"
        eixo_y = "Dia da semana"

    if not dados:
        return None

    fig = go.Figure(
        data=go.Heatmap(
            z=dados,
            x=horas_ordem,
            y=labels_y,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            hovertemplate=f"Métrica: {metrica}<br>{eixo_y}: %{{y}}<br>Hora: %{{x}}<br>{variavel.upper()}: %{{z:.4f}}<extra></extra>",
        )
    )

    fig.update_layout(
        title=titulo,
        xaxis_title="Hora",
        yaxis_title=eixo_y,
        height=500,
        margin=dict(l=80, r=30, t=70, b=40)
    )

    return fig


modo_widget = widgets.ToggleButtons(
    options=["Estações por dia", "Dias por estação"],
    description="Comparação:"
 )

metrica_widget = widgets.ToggleButtons(
    options=["Média", "Desvio padrão"],
    value="Média",
    description="Métrica:"
)

variavel_widget = widgets.Dropdown(
    options=["co", "nox", "no", "no2"],
    value="co",
    description="Poluente:"
 )

contexto_widget = widgets.Dropdown(description="Seleção:")

saida = widgets.Output()


def atualizar_contexto(*args):
    if modo_widget.value == "Estações por dia":
        contexto_widget.options = [(dias_semana[d], d) for d in range(7)]
        contexto_widget.value = 0
        contexto_widget.description = "Dia:"
    else:
        contexto_widget.options = [(nome, idx) for idx, nome in estacoes_usadas.items()]
        contexto_widget.value = 0
        contexto_widget.description = "Estação:"


def atualizar_grafico(*args):
    with saida:
        clear_output(wait=True)
        figura = montar_heatmap(
            modo_widget.value,
            variavel_widget.value,
            contexto_widget.value,
            metrica_widget.value,
        )
        if figura is None:
            display(HTML("<b>Nenhuma tabela desta comparação possui a variável selecionada.</b>"))
        else:
            display(HTML(figura.to_html(include_plotlyjs="cdn", full_html=False)))

modo_widget.observe(atualizar_contexto, names="value")
modo_widget.observe(atualizar_grafico, names="value")
metrica_widget.observe(atualizar_grafico, names="value")
variavel_widget.observe(atualizar_grafico, names="value")
contexto_widget.observe(atualizar_grafico, names="value")

atualizar_contexto()
display(widgets.VBox([widgets.HBox([modo_widget, metrica_widget, variavel_widget, contexto_widget]), saida]))
atualizar_grafico()